# ArcGIS Quick Integration Test

This notebook performs direct, real-connection checks:

1. Load `.env` variables
2. Import ArcGIS Python API
3. Instantiate `GIS`
4. Validate portal connection
5. Instantiate `ImageryLayer`
6. Validate ImageServer/Imagery calls

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Load environment values from repository root .env
repo_root = Path.cwd()
if not (repo_root / ".env").exists() and (repo_root.parent / ".env").exists():
    repo_root = repo_root.parent

env_path = repo_root / ".env"
load_dotenv(env_path)

print(f"Loaded .env from: {env_path}")

portal_url = os.getenv("ARCGIS_PORTAL_URL", "").strip()
image_server_url = "https://mapas.pd.anatel.gov.br/image/rest/services/DEM/ImageServer"
# Optional but important when token is restricted by referer policy
token_referer = os.getenv("ARCGIS_TOKEN_REFERER", "").strip()

print(f"ARCGIS_PORTAL_URL set: {portal_url}")
print(f"ImageServer URL      : {image_server_url}")
print(f"ARCGIS_TOKEN_REFERER : {token_referer or '(not set)'}")

In [ ]:
from arcgis.gis import GIS
from arcgis.raster import ImageryLayer
from arcgis.geometry import Point

print("ArcGIS Python API imports OK")

## Test 1: Instantiate GIS Object

Use **API key authentication** (long-lived token) from `.env` (`ARCGIS_API_KEY`) to create the `GIS` object for portal access tests.

In [ ]:
# Build GIS object using API key authentication (long-lived token)
api_key = os.getenv("ARCGIS_API_KEY", "").strip()

if not portal_url:
    raise ValueError("ARCGIS_PORTAL_URL is required in .env")
if not api_key:
    raise ValueError("ARCGIS_API_KEY is required in .env for Test 1")

gis = GIS(portal_url, api_key=api_key)
auth_mode = "api_key_long_lived_token"

print(f"GIS object created using mode: {auth_mode}")
print("API key loaded from .env (value hidden).")

## Test 2: Validate GIS Connection

Read portal properties and user context to confirm the portal session is working.

In [ ]:
# Test GIS connection by reading portal metadata
portal_props = gis.properties
print("GIS connection OK")
print(f"Portal host    : {portal_props.get('portalHostname', 'unknown')}")
print(f"Portal version : {portal_props.get('currentVersion', 'unknown')}")

# Optional user context
try:
    me = gis.users.me
    print(f"Authenticated as: {getattr(me, 'username', None) or 'N/A'}")
except Exception as exc:
    print(f"User context unavailable: {exc}")

## Test 3: Validate ImageryLayer Connection

Instantiate `ImageryLayer` and test raster access.

If ArcGIS Server returns `499/498`, the cell performs a fallback token exchange (`Portal token -> Server token`) via `generateToken` and validates access with `identify`.

If your environment enforces **token referer restrictions**, set `ARCGIS_TOKEN_REFERER` in `.env` so the fallback requests include the expected `Referer` header.

In [ ]:
# Instantiate ImageryLayer and test ImageServer connectivity
import requests

img = ImageryLayer(image_server_url, gis=gis)
print("ImageryLayer object created")

try:
    # Basic service metadata test via ArcGIS API
    img_props = img.properties
    print("Imagery service metadata OK")
    print(f"Service name: {img_props.get('name', 'unknown')}")

    # Point sample test (identify-like pixel check)
    pt = Point({
        "x": -40.75,
        "y": -19.75,
        "spatialReference": {"wkid": 4326},
    })

    samples = img.get_samples(
        geometry=pt,
        return_first_value_only=True,
        out_fields="*",
    )

    print("Sample query OK")
    print(samples)

except Exception as exc:
    print(f"ArcGIS API call failed: {exc}")
    print("Trying Portal->Server token exchange fallback...")

    # Optional header for tokens restricted by referer policy
    headers = {"Referer": token_referer} if token_referer else {}
    if token_referer:
        print(f"Using Referer header from .env: {token_referer}")

    # 1) Get portal token from GIS connection
    portal_token = gis._con.token
    if not portal_token:
        raise RuntimeError("No portal token available in GIS connection.")

    # 1b) Optional token introspection for debugging privileges/referrer context
    self_url = f"{portal_url.rstrip('/')}/sharing/rest/self"
    self_params = {
        "f": "pjson",
        "token": portal_token,
        "appInfoToken": portal_token,
    }
    self_resp = requests.get(self_url, params=self_params, headers=headers, timeout=30)
    self_resp.raise_for_status()
    self_data = self_resp.json()
    if "error" in self_data:
        print(f"Token self() check returned error: {self_data}")
    else:
        app_info = self_data.get("appInfo", {})
        print("Portal token introspection OK")
        if app_info:
            print(f"App title: {app_info.get('appTitle', 'unknown')}")

    # 2) Exchange portal token for server token
    server_root = "https://mapas.pd.anatel.gov.br/image"
    gen_url = f"{portal_url.rstrip('/')}/sharing/rest/generateToken"
    gen_payload = {
        "f": "json",
        "token": portal_token,
        "serverUrl": server_root,
    }
    gen_resp = requests.post(gen_url, data=gen_payload, headers=headers, timeout=30)
    gen_resp.raise_for_status()
    gen_data = gen_resp.json()

    if "error" in gen_data:
        raise RuntimeError(
            "generateToken failed. "
            "If message contains 'Invalid Referer', configure ARCGIS_TOKEN_REFERER in .env. "
            f"Response: {gen_data}"
        )

    server_token = gen_data.get("token")
    if not server_token:
        raise RuntimeError(f"No server token returned: {gen_data}")

    print("Server token acquired (value hidden).")

    # 3) Validate with direct identify call
    identify_url = f"{image_server_url}/identify"
    identify_params = {
        "geometry": "-40.75,-19.75",
        "geometryType": "esriGeometryPoint",
        "sr": "4326",
        "returnCatalogItems": "false",
        "returnGeometry": "false",
        "f": "pjson",
        "token": server_token,
    }

    id_resp = requests.get(identify_url, params=identify_params, headers=headers, timeout=30)
    id_resp.raise_for_status()
    id_data = id_resp.json()

    if "error" in id_data:
        raise RuntimeError(f"identify failed: {id_data}")

    print("Fallback identify OK")
    print(id_data)